In [2]:
import os
import glob
import polars as pl

# ==========================================
# 1. CONFIGURATION PATHS & CONSTANTS
# ==========================================
DATA_DIR = r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds"
TARGET_MAP = "dust2"
MATCHES_TO_PROCESS = 10

def find_dust2_bounds():
    print(f"Searching for {MATCHES_TO_PROCESS} matches on map '{TARGET_MAP}'...\n")
    
    # Initialize global boundary tracking variables
    global_min_x = float('inf')
    global_max_x = float('-inf')
    global_min_y = float('inf')
    global_max_y = float('-inf')
    
    matches_processed = 0
    
    # ==========================================
    # 2. FILE DISCOVERY & PARSING
    # ==========================================
    # Find all header files recursively, ignoring exact extensions
    search_pattern = os.path.join(DATA_DIR, "**", "header*")
    header_files = glob.glob(search_pattern, recursive=True)
    
    for header_path in header_files:
        if not os.path.isfile(header_path):
            continue
            
        match_dir = os.path.dirname(header_path)
        match_id = os.path.basename(match_dir)
        
        # Attempt to read the header file to verify the map
        try:
            df_header = pl.read_parquet(header_path)
            raw_map_name = str(df_header["map_name"][0]).lower()
            map_name = raw_map_name.replace("de_", "")  # Normalize 'de_dust2' to 'dust2'
        except Exception:
            # Skip corrupted/invalid files silently
            continue
            
        # ==========================================
        # 3. WEAPON FIRE PARSING & BOUNDARY EXTRACTION
        # ==========================================
        if map_name == TARGET_MAP:
            wf_pattern = os.path.join(match_dir, "weapon_fire*")
            wf_files = glob.glob(wf_pattern)
            
            if not wf_files:
                continue
                
            wf_path = wf_files[0]
            
            try:
                df_wf = pl.read_parquet(wf_path)
                
                # Verify columns exist
                if "player_x_pos" in df_wf.columns and "player_y_pos" in df_wf.columns:
                    # Find min/max for this specific match
                    local_min_x = df_wf["player_x_pos"].min()
                    local_max_x = df_wf["player_x_pos"].max()
                    local_min_y = df_wf["player_y_pos"].min()
                    local_max_y = df_wf["player_y_pos"].max()
                    
                    # Update the global map bounds
                    global_min_x = min(global_min_x, local_min_x)
                    global_max_x = max(global_max_x, local_max_x)
                    global_min_y = min(global_min_y, local_min_y)
                    global_max_y = max(global_max_y, local_max_y)
                    
                    matches_processed += 1
                    print(f"[{matches_processed}/{MATCHES_TO_PROCESS}] Processed bounds for match: {match_id}")
                    
                    # Stop once we hit our target sample size
                    if matches_processed >= MATCHES_TO_PROCESS:
                        break
            except Exception as e:
                print(f"Error reading weapon_fire for match {match_id}: {e}")
                continue

    # ==========================================
    # 4. RESULTS DISPLAY
    # ==========================================
    if matches_processed > 0:
        print("\n" + "="*50)
        print(f" EMPIRICAL MAP BOUNDS FOR: {TARGET_MAP.upper()}")
        print(f" Matches analyzed: {matches_processed}")
        print("="*50)
        print(f" x_min : {global_min_x:>10.2f}")
        print(f" x_max : {global_max_x:>10.2f}")
        print(f" y_min : {global_min_y:>10.2f}")
        print(f" y_max : {global_max_y:>10.2f}")
        print("="*50)
        
        # Print dictionary format for easy copy-pasting into your main script
        print("\nDictionary Output format:")
        print(f"MAP_BOUNDS_Dust2 = {{'x_min': {global_min_x}, 'x_max': {global_max_x}, 'y_min': {global_min_y}, 'y_max': {global_max_y}}}")
    else:
        print(f"No valid matches found for map '{TARGET_MAP}'.")

if __name__ == "__main__":
    find_dust2_bounds()

Searching for 10 matches on map 'dust2'...

[1/10] Processed bounds for match: 0826b896-52ee-4bb5-9d00-165e599fdaee
[2/10] Processed bounds for match: 102860e5-87ed-4b47-a129-27ba0b39267f
[3/10] Processed bounds for match: 12290dd9-63bd-41a5-8408-b03fe72af2c4
[4/10] Processed bounds for match: 15b65468-651a-4ce3-b3e5-7755623f3b4d
[5/10] Processed bounds for match: 23aaf423-615d-45c4-82a0-837a782d831f
[6/10] Processed bounds for match: 349ef7c0-77eb-4777-9d4a-80d8d2d65e6c
[7/10] Processed bounds for match: 36277387-28ba-4706-bc2a-fb2ba6da5365
[8/10] Processed bounds for match: 3a3e27ea-676f-4959-9599-8b7d11a25763
[9/10] Processed bounds for match: 4b2a7011-e466-48fa-a117-4870cb7a1910
[10/10] Processed bounds for match: 4e0cde01-f4b6-4d69-a16d-16c22dceb48d

 EMPIRICAL MAP BOUNDS FOR: DUST2
 Matches analyzed: 10
 x_min :   -2163.60
 x_max :    1787.97
 y_min :   -1163.97
 y_max :    3077.69

Dictionary Output format:
MAP_BOUNDS_Dust2 = {'x_min': -2163.597412109375, 'x_max': 1787.97265625,